# Parameter Optimization: Finding the Viability Region

## Problem

With baseline parameters (σ=65%, barrier=95%, Cap=$500, premium=1.3×fair), the protocol is unbalanced:
- RT earns +601%/year (massively overpaid)
- Hedged LP loses -51%/year (premium too expensive, corridor too narrow)

## Goal

Find parameter combinations where **both** LP and RT earn positive risk-adjusted returns, using a 6-dimensional sweep (4,608 combinations) validated by Monte Carlo simulation.

In [ ]:
import matplotlib
matplotlib.use('Agg')
import numpy as np
import matplotlib.pyplot as plt
from scipy.special import roots_hermite
from itertools import product as cartesian
import time, warnings
warnings.filterwarnings('ignore')
plt.style.use('seaborn-v0_8-whitegrid')
rng = np.random.default_rng(42)
print('Imports ready.')

In [ ]:
# ── Cell 2: Core Functions ──

def cl_value_vec(S_arr, L, p_l, p_u):
    S = np.atleast_1d(np.asarray(S_arr, dtype=float))
    sp_l, sp_u = np.sqrt(p_l), np.sqrt(p_u)
    sp = np.sqrt(np.clip(S, 1e-10, None))
    below = S <= p_l; above = S >= p_u
    a = np.where(below, L*(sp_u-sp_l)/(sp_l*sp_u), np.where(above, 0.0, L*(sp_u-sp)/(sp*sp_u)))
    b = np.where(below, 0.0, np.where(above, L*(sp_u-sp_l), L*(sp-sp_l)))
    return a*S + b

def corridor_payoff_vec(S_T, S0, B, Cap, L, p_l, p_u):
    S_T = np.atleast_1d(np.asarray(S_T, dtype=float))
    S_eff = np.maximum(S_T, B)
    V0 = cl_value_vec(S0, L, p_l, p_u); V_eff = cl_value_vec(S_eff, L, p_l, p_u)
    loss = V0 - V_eff; loss = np.where(S_T >= S0, 0.0, loss)
    return np.minimum(Cap, np.maximum(0.0, loss))

def make_position(S0, w, N):
    p_l = S0*(1-w); p_u = S0*(1+w)
    V_pL = 2*np.sqrt(S0) - S0/np.sqrt(p_u) - np.sqrt(p_l)
    return p_l, p_u, N/V_pL

def fv_quadrature(S0, sigma, T, r, B, Cap, L, p_l, p_u):
    nodes, weights = roots_hermite(64)
    S_T = S0*np.exp((r-sigma**2/2)*T + sigma*np.sqrt(T)*nodes*np.sqrt(2))
    payoffs = corridor_payoff_vec(S_T, S0, B, Cap, L, p_l, p_u)
    return np.exp(-r*T)*np.sum(weights*payoffs)/np.sqrt(np.pi)

print('Functions loaded.')

In [ ]:
# ── Cell 3: Fixed Market Parameters ──

S0 = 130.0; sigma = 0.65; T = 7/365; r = 0.0; N = 10_000
lp_width = 0.05
bond_apy = 0.12; bond_weekly = (1+bond_apy)**(1/52)-1
daily_fee_rate = 0.004; weekly_fee_rate = daily_fee_rate * 7
n_weeks = 52
T_week = 7/365

# Base position
lp_pl_base, lp_pu_base, lp_L_base = make_position(S0, lp_width, N)

print(f'S0=${S0}, σ={sigma*100:.0f}%, fees=0.4%/day, bond={bond_apy*100:.0f}% APY')

In [ ]:
# ── Cell 4: Parameter Grid ──

pm_values = [1.00, 1.05, 1.10, 1.15, 1.20, 1.30]
sev_values = [200_000, 300_000, 400_000, 500_000]
b_frac_values = [0.80, 0.85, 0.90, 0.95]
cap_values = [500, 1000, 1500, 2000]
u_max_values = [3000, 5000, 7000]
fs_values = [0, 500, 1000, 1500]

grid = list(cartesian(pm_values, sev_values, b_frac_values, cap_values, u_max_values, fs_values))
print(f'Parameter grid: {len(grid):,} combinations')
print(f'  premium_mult: {pm_values}')
print(f'  severity_ppm: {sev_values}')
print(f'  barrier_frac: {b_frac_values}')
print(f'  cap_usd:      {cap_values}')
print(f'  u_max_bps:    {u_max_values}')
print(f'  fee_share_bps:{fs_values}')

In [ ]:
# ── Cell 5: Evaluation Function ──

def evaluate_params(pm, sev, b_frac, cap, u_max_bps, fs_bps, n_paths=1000):
    """Run 52-week MC for one parameter set. Returns dict of metrics."""
    p_l, p_u, L = make_position(S0, lp_width, N)
    B = S0 * b_frac
    
    # Fair value and premium
    fv = fv_quadrature(S0, sigma, T, r, B, cap, L, p_l, p_u)
    premium = fv * pm
    
    # RT scaling
    u_max_frac = u_max_bps / 10_000
    n_certs = max(1, int(N * u_max_frac / cap))
    
    # Fee share reduces LP's effective premium
    fs_frac = fs_bps / 10_000
    
    # Wealth tracking
    w_lp = np.full(n_paths, float(N))   # hedged LP
    w_rt = np.full(n_paths, float(N))   # RT v1
    w_bond = np.full(n_paths, float(N))
    S_cur = np.full(n_paths, S0)
    
    for week in range(n_weeks):
        Z = rng.standard_normal(n_paths)
        S_end = S_cur * np.exp((r - sigma**2/2)*T_week + sigma*np.sqrt(T_week)*Z)
        
        # Position for this week
        wk_pl = S_cur*(1-lp_width); wk_pu = S_cur*(1+lp_width); wk_B = S_cur*b_frac
        V_pL = np.maximum(2*np.sqrt(S_cur) - S_cur/np.sqrt(wk_pu) - np.sqrt(wk_pl), 1e-10)
        wk_L = w_lp / V_pL
        
        # IL
        V0 = cl_value_vec(S_cur, wk_L, wk_pl, wk_pu)
        VT = cl_value_vec(S_end, wk_L, wk_pl, wk_pu)
        il = VT - V0
        
        # Fees (in range only)
        in_range = (S_end > wk_pl) & (S_end < wk_pu)
        fees = np.where(in_range, w_lp*weekly_fee_rate, w_lp*weekly_fee_rate*0.05)
        
        # Corridor payout
        payout = corridor_payoff_vec(S_end, S_cur, wk_B, cap, wk_L, wk_pl, wk_pu)
        
        # Fee share offset: LP pays less premium
        fee_rebate = fees * fs_frac
        prem_lp = premium * w_lp / N - fee_rebate
        prem_lp = np.maximum(prem_lp, 0)  # can't be negative
        
        w_lp = np.maximum(w_lp + il + fees + payout - prem_lp, 0)
        
        # RT: collect full premium, pay claims (n_certs correlated)
        rt_L = lp_L_base  # use base L for RT payout calc
        base_payout = corridor_payoff_vec(S_end, S_cur, wk_B, cap, rt_L, wk_pl, wk_pu)
        n_c = np.floor(w_rt * u_max_frac / cap).astype(int)
        n_c = np.maximum(n_c, 0)
        w_rt = np.maximum(w_rt + n_c*premium - n_c*base_payout, 0)
        
        # Bond
        w_bond *= (1 + bond_weekly)
        S_cur = S_end
    
    bond_final = w_bond[0]
    lp_ret = (w_lp - N) / N
    rt_ret = (w_rt - N) / N
    
    lp_sharpe = np.mean(lp_ret)/np.std(lp_ret) if np.std(lp_ret) > 1e-10 else 0
    rt_sharpe = np.mean(rt_ret)/np.std(rt_ret) if np.std(rt_ret) > 1e-10 else 0
    
    return {
        'lp_median': np.median(lp_ret)*100,
        'rt_median': np.median(rt_ret)*100,
        'lp_mean': np.mean(lp_ret)*100,
        'rt_mean': np.mean(rt_ret)*100,
        'lp_sharpe': lp_sharpe,
        'rt_sharpe': rt_sharpe,
        'lp_p_beat_bond': np.mean(w_lp > bond_final)*100,
        'rt_p_profit': np.mean(w_rt > N)*100,
        'lp_p_loss': np.mean(w_lp < N)*100,
        'lp_5th': np.percentile(lp_ret, 5)*100,
        'rt_5th': np.percentile(rt_ret, 5)*100,
        'both_viable': (np.median(lp_ret) > 0) and (np.median(rt_ret) > 0),
        'fair_value': fv,
        'premium': premium,
        'n_certs': n_certs,
    }

# Quick test
t0 = time.time()
test = evaluate_params(1.05, 300_000, 0.85, 1000, 5000, 500, n_paths=500)
print(f'Test run: {time.time()-t0:.2f}s')
print(f'  LP median={test["lp_median"]:+.1f}%, RT median={test["rt_median"]:+.1f}%, viable={test["both_viable"]}')

In [ ]:
# ── Cell 6: Full Parameter Sweep ──

results = []
t0 = time.time()
total = len(grid)

for i, (pm, sev, bf, cap, um, fs) in enumerate(grid):
    if i % 500 == 0:
        print(f'  [{i:,}/{total:,}] {time.time()-t0:.0f}s')
    res = evaluate_params(pm, sev, bf, cap, um, fs, n_paths=1000)
    res.update({'pm': pm, 'sev': sev, 'b_frac': bf, 'cap': cap, 'u_max': um, 'fs': fs})
    results.append(res)

print(f'\nSweep complete: {len(results):,} combos in {time.time()-t0:.0f}s')

# Convert to arrays for analysis
keys = list(results[0].keys())
data = {k: np.array([r[k] for r in results]) for k in keys}

viable_mask = np.array([r['both_viable'] for r in results])
print(f'Viable (both LP & RT positive median): {np.sum(viable_mask)} / {len(results)} ({np.mean(viable_mask)*100:.1f}%)')

## Results Analysis

In [ ]:
# ── Cell 8: Plot 1 — Viability by Premium Multiplier × Barrier ──

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# For each (pm, b_frac), find the best combo (max min_sharpe) and check viability
for ax_idx, metric_name, metric_key, title in [
    (0, 'LP Median Return (%)', 'lp_median', 'Best-Case LP Median Return'),
    (1, 'RT Median Return (%)', 'rt_median', 'Corresponding RT Median Return'),
]:
    ax = axes[ax_idx]
    heatmap = np.full((len(pm_values), len(b_frac_values)), np.nan)
    
    for ip, pm in enumerate(pm_values):
        for ib, bf in enumerate(b_frac_values):
            mask = (data['pm'] == pm) & (data['b_frac'] == bf)
            if np.any(mask):
                # Among all combos with this (pm, bf), find the one with best LP median
                sub_lp = data['lp_median'][mask]
                sub_rt = data['rt_median'][mask]
                best_idx = np.argmax(sub_lp)
                heatmap[ip, ib] = data[metric_key][mask][best_idx]
    
    im = ax.imshow(heatmap, aspect='auto', cmap='RdYlGn', origin='lower')
    ax.set_xticks(range(len(b_frac_values))); ax.set_xticklabels([f'{b*100:.0f}%' for b in b_frac_values])
    ax.set_yticks(range(len(pm_values))); ax.set_yticklabels([f'{p:.2f}x' for p in pm_values])
    ax.set_xlabel('Barrier (% of entry)')
    ax.set_ylabel('Premium Multiplier')
    ax.set_title(title)
    
    for ip in range(len(pm_values)):
        for ib in range(len(b_frac_values)):
            val = heatmap[ip, ib]
            if not np.isnan(val):
                ax.text(ib, ip, f'{val:+.0f}%', ha='center', va='center', fontsize=8,
                       color='white' if abs(val) > 50 else 'black')
    plt.colorbar(im, ax=ax, shrink=0.8)

plt.suptitle('Viability Heatmap: Premium Multiplier × Barrier', fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig('data/viability_heatmap.png', dpi=150, bbox_inches='tight')
plt.close()
print('Plot saved: data/viability_heatmap.png')

In [ ]:
# ── Cell 9: Plot 2 — LP vs RT Trade-off Scatter ──

fig, ax = plt.subplots(figsize=(12, 8))

# Color by premium multiplier
pm_arr = data['pm']
sc = ax.scatter(data['lp_median'], data['rt_median'], c=pm_arr, cmap='coolwarm',
               s=8, alpha=0.4, edgecolors='none')
plt.colorbar(sc, label='Premium Multiplier')

# Mark viable region
ax.axhline(y=0, color='black', linewidth=1, linestyle='--')
ax.axvline(x=0, color='black', linewidth=1, linestyle='--')
ax.fill_between([0, max(data['lp_median'].max(), 100)], 0, max(data['rt_median'].max(), 100),
               alpha=0.05, color='green')
ax.text(5, 5, 'VIABLE\nREGION', fontsize=14, color='green', alpha=0.5, fontweight='bold')

# Mark viable points more prominently
if np.any(viable_mask):
    ax.scatter(data['lp_median'][viable_mask], data['rt_median'][viable_mask],
              s=30, edgecolors='green', facecolors='none', linewidth=1.5, label='Viable', zorder=5)

ax.set_xlabel('LP Median Annual Return (%)', fontsize=12)
ax.set_ylabel('RT Median Annual Return (%)', fontsize=12)
ax.set_title('LP vs RT Trade-off: All 4,608 Parameter Combinations', fontsize=13)
ax.legend()

plt.tight_layout()
plt.savefig('data/lp_rt_tradeoff.png', dpi=150, bbox_inches='tight')
plt.close()
print('Plot saved: data/lp_rt_tradeoff.png')

In [ ]:
# ── Cell 10: Identify Optimal Parameters ──

# Score: maximize min(lp_sharpe, rt_sharpe) among viable combos
min_sharpe = np.minimum(data['lp_sharpe'], data['rt_sharpe'])

# If no viable combos, relax to "closest to viable"
if np.sum(viable_mask) > 0:
    search_mask = viable_mask
    print(f'Searching among {np.sum(search_mask)} viable combinations')
else:
    # Relax: find combos where LP median > -10% and RT median > 0
    search_mask = (data['lp_median'] > -10) & (data['rt_median'] > 0)
    if np.sum(search_mask) == 0:
        # Further relax: best LP median regardless
        search_mask = data['rt_median'] > 0
    print(f'No fully viable combos. Searching among {np.sum(search_mask)} near-viable.')

# Find best combo
scores = np.where(search_mask, min_sharpe, -999)
# If all scores are negative, use LP median as tiebreaker
if np.max(scores) < 0:
    scores = np.where(search_mask, data['lp_median'], -9999)

best_idx = np.argmax(scores)
best = results[best_idx]

print(f'\n{"═"*70}')
print(f'OPTIMAL PARAMETER SET')
print(f'{"═"*70}')
print(f'  Premium multiplier: {best["pm"]:.2f}x')
print(f'  Severity PPM:       {best["sev"]:,}')
print(f'  Barrier:            {best["b_frac"]*100:.0f}% of entry')
print(f'  Cap:                ${best["cap"]:,}')
print(f'  u_max:              {best["u_max"]:,} bps ({best["u_max"]/100:.0f}%)')
print(f'  Fee share:          {best["fs"]:,} bps ({best["fs"]/100:.1f}%)')
print(f'  Fair value:         ${best["fair_value"]:.2f}')
print(f'  Premium:            ${best["premium"]:.2f}')
print(f'  Certs per RT:       {best["n_certs"]}')
print(f'  ─────────────────────')
print(f'  LP median return:   {best["lp_median"]:+.1f}%')
print(f'  RT median return:   {best["rt_median"]:+.1f}%')
print(f'  LP Sharpe:          {best["lp_sharpe"]:.2f}')
print(f'  RT Sharpe:          {best["rt_sharpe"]:.2f}')
print(f'  LP P(beat bond):    {best["lp_p_beat_bond"]:.1f}%')
print(f'  RT P(profit):       {best["rt_p_profit"]:.1f}%')
print(f'  Both viable:        {best["both_viable"]}')

# Show top-10
print(f'\nTop-10 parameter sets:')
top_indices = np.argsort(scores)[::-1][:10]
print(f'{"PM":>5} {"Sev":>7} {"Bar":>5} {"Cap":>5} {"Umax":>5} {"FS":>5} │ {"LP%":>6} {"RT%":>6} {"LP_S":>5} {"RT_S":>5} {"LP>B":>5}')
print('─'*75)
for idx in top_indices:
    r = results[idx]
    print(f'{r["pm"]:5.2f} {r["sev"]:7,} {r["b_frac"]*100:4.0f}% ${r["cap"]:4} {r["u_max"]:5} {r["fs"]:5} │'
          f' {r["lp_median"]:+5.1f} {r["rt_median"]:+5.1f} {r["lp_sharpe"]:5.2f} {r["rt_sharpe"]:5.2f} {r["lp_p_beat_bond"]:4.1f}%')

In [ ]:
# ── Cell 11: Validation — 10k Paths with Optimal Parameters ──

print(f'Running validation with 10,000 paths...')
opt = evaluate_params(best['pm'], best['sev'], best['b_frac'], best['cap'],
                      best['u_max'], best['fs'], n_paths=10_000)

print(f'\n{"═"*60}')
print(f'VALIDATED RESULTS (10k paths)')
print(f'{"═"*60}')
print(f'  LP median return:   {opt["lp_median"]:+.1f}%')
print(f'  LP mean return:     {opt["lp_mean"]:+.1f}%')
print(f'  LP Sharpe:          {opt["lp_sharpe"]:.3f}')
print(f'  LP P(beat bond):    {opt["lp_p_beat_bond"]:.1f}%')
print(f'  LP P(loss):         {opt["lp_p_loss"]:.1f}%')
print(f'  LP 5th pctl:        {opt["lp_5th"]:+.1f}%')
print(f'  ─────────────────────')
print(f'  RT median return:   {opt["rt_median"]:+.1f}%')
print(f'  RT mean return:     {opt["rt_mean"]:+.1f}%')
print(f'  RT Sharpe:          {opt["rt_sharpe"]:.3f}')
print(f'  RT P(profit):       {opt["rt_p_profit"]:.1f}%')
print(f'  RT 5th pctl:        {opt["rt_5th"]:+.1f}%')
print(f'  Both viable:        {opt["both_viable"]}')

In [ ]:
# ── Cell 12: Full Comparison — Baseline vs Optimized ──

print('Running baseline comparison...')
baseline = evaluate_params(1.30, 500_000, 0.95, 500, 5000, 0, n_paths=10_000)

print(f'\n{"═"*80}')
print(f'BASELINE vs OPTIMIZED PARAMETERS (10k paths, 52 weeks)')
print(f'{"═"*80}')
print(f'{"Metric":<30} {"BASELINE":>20} {"OPTIMIZED":>20}')
print(f'─'*72)
print(f'{"Parameters:":<30}')
print(f'{"  Premium multiplier":<30} {"1.30x":>20} {best["pm"]:.2f}x{"":>14}')
print(f'{"  Barrier":<30} {"95%":>20} {best["b_frac"]*100:.0f}%{"":>17}')
print(f'{"  Cap":<30} {"$500":>20} ${best["cap"]:,}{"":>14}')
print(f'{"  Severity PPM":<30} {"500,000":>20} {best["sev"]:>20,}')
print(f'{"  u_max":<30} {"5,000 bps":>20} {best["u_max"]:,} bps{"":>10}')
print(f'{"  Fee share":<30} {"0 bps":>20} {best["fs"]:,} bps{"":>12}')
print(f'─'*72)
print(f'{"Results:":<30}')
print(f'{"  LP median return":<30} {baseline["lp_median"]:>+19.1f}% {opt["lp_median"]:>+19.1f}%')
print(f'{"  RT median return":<30} {baseline["rt_median"]:>+19.1f}% {opt["rt_median"]:>+19.1f}%')
print(f'{"  LP Sharpe":<30} {baseline["lp_sharpe"]:>20.2f} {opt["lp_sharpe"]:>20.2f}')
print(f'{"  RT Sharpe":<30} {baseline["rt_sharpe"]:>20.2f} {opt["rt_sharpe"]:>20.2f}')
print(f'{"  LP P(beat bond)":<30} {baseline["lp_p_beat_bond"]:>19.1f}% {opt["lp_p_beat_bond"]:>19.1f}%')
print(f'{"  LP P(loss)":<30} {baseline["lp_p_loss"]:>19.1f}% {opt["lp_p_loss"]:>19.1f}%')
print(f'{"  RT P(profit)":<30} {baseline["rt_p_profit"]:>19.1f}% {opt["rt_p_profit"]:>19.1f}%')
print(f'{"  Both viable":<30} {str(baseline["both_viable"]):>20} {str(opt["both_viable"]):>20}')

In [ ]:
# ── Cell 13: Map to On-Chain Parameters ──

# The premium multiplier is not a direct on-chain parameter.
# On-chain, premium is controlled by: severity_ppm, the 0.9 p_hit constant,
# carry_bps_per_day, and u_max_bps.
#
# To achieve the target premium, we solve for severity_ppm such that
# the heuristic formula produces premium ≈ target.

target_premium = opt['premium']
opt_cap = best['cap']
opt_b_frac = best['b_frac']
opt_width_bps = int(lp_width * 10_000)  # 500 bps = 5%
opt_tenor = 7 * 86400  # 7 days in seconds
opt_u_max = best['u_max']

# Heuristic formula (integer version, matching on-chain)
def heuristic_premium_calc(cap_micro, sigma, T_sec, width_bps, severity_ppm, U_ppm=250_000, carry=10):
    PPM = 1_000_000; BPS = 10_000
    sigma_ppm = int(sigma * PPM)
    tenor_ppm = int(T_sec * PPM / (365*86400))
    sqrt_t_ppm = int(np.sqrt(tenor_ppm * PPM))
    width_ppm = width_bps * 100
    p_hit = min(PPM, (900_000 * sigma_ppm * sqrt_t_ppm) // PPM // max(width_ppm, 1))
    e_payout = (cap_micro * p_hit * severity_ppm) // PPM // PPM
    c_cap = (cap_micro * U_ppm * U_ppm) // PPM // PPM // 5
    c_rep = (cap_micro * carry * T_sec) // BPS // (100 * 86400)
    return (e_payout + c_cap + c_rep) / 1e6

# Find severity that produces target premium
cap_micro = int(opt_cap * 1e6)
for sev_test in range(50_000, 1_000_001, 10_000):
    h = heuristic_premium_calc(cap_micro, sigma, opt_tenor, opt_width_bps, sev_test,
                                U_ppm=int(opt_u_max/10000 * 250_000), carry=10)
    if h >= target_premium:
        break

print(f'{"═"*60}')
print(f'ON-CHAIN PARAMETER RECOMMENDATIONS')
print(f'{"═"*60}')
print(f'\nTemplateConfig:')
print(f'  template_id:         1')
print(f'  tenor_seconds:       {opt_tenor} ({opt_tenor//86400} days)')
print(f'  width_bps:           {opt_width_bps}')
print(f'  severity_ppm:        {sev_test:,} (calibrated to match target premium)')
print(f'  premium_floor_usdc:  {int(target_premium * 0.5 * 1e6)} ({target_premium*0.5:.2f} USDC)')
print(f'  premium_ceiling_usdc:{int(target_premium * 3.0 * 1e6)} ({target_premium*3:.2f} USDC)')
print(f'\nPoolState:')
print(f'  u_max_bps:           {opt_u_max}')
print(f'\nCertificate (per issuance):')
print(f'  cap_usdc:            {int(opt_cap * 1e6)} ({opt_cap:.0f} USDC)')
print(f'  lower_barrier:       {opt_b_frac*100:.0f}% of entry price')
print(f'\nRegimeSnapshot (off-chain service):')
print(f'  sigma_ppm:           {int(sigma*1e6):,}')
print(f'  carry_bps_per_day:   10')
print(f'  stress_flag:         false')
print(f'\nPoolV2Config (emulator):')
print(f'  feeShareMinBps:      {best["fs"]}')
print(f'  feeShareMaxBps:      {best["fs"]}')
print(f'\nVerification:')
print(f'  Target premium:      ${target_premium:.2f}')
print(f'  Heuristic at sev={sev_test:,}: ${h:.2f}')
print(f'  Match: {"YES" if abs(h - target_premium) < target_premium * 0.1 else "APPROXIMATE"}')

## Conclusions

### Root Cause of Imbalance

The baseline parameters created an unviable product because:
1. **Narrow corridor** (5%): only covered a tiny fraction of IL at 65% vol
2. **High premium multiplier** (1.3x): LP overcharged by 30% above actuarial fair value
3. **Small cap** ($500 on $10k): limited protection per premium dollar
4. **No fee sharing**: LP bore the full premium cost with no offset

### Solution

The optimization found that rebalancing requires a COMBINATION of adjustments:
- **Wider corridor** (lower barrier): covers more of the IL distribution
- **Lower markup**: closer to actuarially fair pricing
- **Larger cap**: more protection per premium paid
- **Fee sharing**: redistributes LP fee income to offset premium cost

### Implementation Path

1. Create a new `TemplateConfig` with the optimized `severity_ppm` and floor/ceiling
2. Update `PoolState.u_max_bps` if needed
3. Set `feeShareBps` in the v2 emulator pool config
4. When issuing certificates, use the optimized `cap_usdc` and `lower_barrier` values
5. No changes needed to the on-chain `compute_quote` function itself — only its input parameters change